# 05 — Model Comparison: Does Anything Beat the Baseline?

**Project:** 30-Day Readmission Risk in Diabetic Inpatients
**Notebook 5 of 6**

---

Notebook 04 ended with a result I did not expect and do not much like. At a fixed
review budget of 5,486 patients, my 37-feature logistic regression caught 1,019 of
2,179 readmissions. Ranking patients by a single integer — how many times they had
been admitted in the past year — caught 972. The entire model bought 47 extra
readmissions over a number a registrar can read off the notes in five seconds.

I argued there that this is a finding about the data rather than a failure of the
model. This notebook is where I have to prove that rather than assert it. One model
performing near a heuristic could just be a weak model. If three different models
all land in the same place, that is a property of the dataset.

So the question here is specific, and it is not "which model scores highest":

> **Does any model beat prior inpatient admissions alone by more than the 2.2
> percentage points the logistic regression manages?**

Three models get tested — the logistic baseline from notebook 03, a gradient
boosting model, and a deliberately stripped-back five-feature model. Each is judged
on catches at a fixed review budget, not on AUC, because AUC is not what a
discharge team experiences. I also run the budget sweep I deferred from notebook 04,
since the 2.2-point gap was measured at one budget and may not hold at others.

No new feature engineering. Same data, same split, same preprocessing — only the
model changes, so that any difference is attributable to the model and nothing else.

---
## 0. Setup, and rebuilding notebook 03's exact split

In [1]:
# Imports and settings, matched to notebook 03 so the split reproduces exactly
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from statsmodels.stats.proportion import proportion_confint

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

RANDOM_STATE = 42
DATA_DIR = "../data"

Notebook 03 loaded the analysis file from notebook 02 and then added two things on
top of it before splitting: it split diabetes by complication in the secondary and
tertiary diagnosis columns, and it turned each lab column into a measurement flag
plus a result category. Those transforms are not saved anywhere, they only exist as
code, so I have to repeat them here or my feature matrix will not match the one the
saved pipeline expects.

In [2]:
# Load the analysis dataset from notebook 02, then re-apply the transforms notebook 03 added before splitting
df = pd.read_csv(f"{DATA_DIR}/processed/cohort_eda.csv", low_memory=False)

def diabetes_detail(code):
    s = str(code)
    if len(s) < 6:
        return "Diabetes, unspecified"
    return "Diabetes, uncomplicated" if s[4] == "0" else "Diabetes, complicated"

# notebook 03: apply the diabetes complication split to the secondary and tertiary diagnoses
for col in ["diag_2", "diag_3"]:
    grp = col + "_group"
    mask = df[grp] == "Diabetes"
    df.loc[mask, grp] = df.loc[mask, col].apply(diabetes_detail)

# notebook 03: columns that never enter the model
EXCLUDE = {"encounter_id", "patient_nbr", "readmitted", "target",
           "diag_1", "diag_2", "diag_3", "under_20"}
FEATURES = [c for c in df.columns if c not in EXCLUDE]

# notebook 03: the ID columns are codes, not quantities, so they are categorical and must be strings
CODED_CATEGORICAL = ["admission_type_id", "discharge_disposition_id", "admission_source_id"]
NUMERIC = [c for c in FEATURES
           if pd.api.types.is_numeric_dtype(df[c]) and c not in CODED_CATEGORICAL]
CATEGORICAL = [c for c in FEATURES if c not in NUMERIC]

for c in CODED_CATEGORICAL:
    df[c] = df[c].astype(str)

# notebook 03: split each lab column into a measurement flag and a result category
for col in ["A1Cresult", "max_glu_serum"]:
    flag = col.lower().replace("result", "") + "_measured"
    df[flag] = df[col].notna().astype(int)
    df[col] = df[col].fillna("not_measured")
NUMERIC += ["a1c_measured", "max_glu_serum_measured"]

print(f"Numeric: {len(NUMERIC)}   Categorical: {len(CATEGORICAL)}   Total: {len(NUMERIC) + len(CATEGORICAL)}")

Numeric: 10   Categorical: 27   Total: 37


In [3]:
# notebook 03: collapse categorical levels seen fewer than 100 times, before the split so both sides share the same levels
MIN_LEVEL_N = 100

def collapse_rare(series, min_n=MIN_LEVEL_N, label="Other_rare"):
    vc = series.value_counts(dropna=False)
    keep = set(vc[vc >= min_n].index)
    return series.where(series.isin(keep), label)

for col in CATEGORICAL:
    df[col] = collapse_rare(df[col].fillna("Unknown"))

print(f"Collapsed {len(CATEGORICAL)} categorical columns at a minimum of {MIN_LEVEL_N} per level")
print(f"Encounters: {len(df):,}  |  Base rate: {df['target'].mean():.1%}")

Collapsed 27 categorical columns at a minimum of 100 per level
Encounters: 99,316  |  Base rate: 11.4%


Now the split. This is the part that has to be exactly right. Every comparison in
this notebook is against the 1,019 catches from notebook 04, and that number is
only meaningful if I am scoring the same patients. Reproducing the seed is not
enough on its own — I want the encounter IDs checked one by one against the file
notebook 03 saved, so that if anything has drifted I find out now rather than after
fitting three models.

In [4]:
# Rebuild notebook 03's split and prove it is the same test set, not merely the same size
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, df["target"], groups=df["patient_nbr"]))

train = df.iloc[train_idx].copy()
test  = df.iloc[test_idx].copy()

nb04 = pd.read_csv(f"{DATA_DIR}/processed/test_predictions.csv")

assert len(test) == len(nb04), f"size mismatch: {len(test):,} vs {len(nb04):,}"
assert set(test["encounter_id"]) == set(nb04["encounter_id"]), \
    "SPLIT DOES NOT MATCH notebook 04 - every comparison below would be invalid"

print(f"Train: {len(train):,} encounters, {train['patient_nbr'].nunique():,} patients")
print(f"Test:  {len(test):,} encounters, {test['patient_nbr'].nunique():,} patients")
print("\nTest set matches notebook 04 encounter-for-encounter.")

Train: 79,682 encounters, 55,981 patients
Test:  19,634 encounters, 13,996 patients

Test set matches notebook 04 encounter-for-encounter.


---
## 1. Why this notebook compares catches, not AUC

The standard way to end a project like this is a leaderboard. Fit five models,
report AUC for each, declare the highest one the winner, imply that picking it was
clever. I do not want to do that, for two reasons.

The first is that AUC does not describe anything a ward experiences. It is the
probability that a randomly chosen readmitted patient is ranked above a randomly
chosen non-readmitted one. Nobody on a discharge round is doing that. What they are
doing is working through a list of fixed length, so the question that matters is how
many readmissions land on that list.

The second is that AUC would hide the finding from notebook 04. Ranking by prior
inpatient admissions alone is a rule, not a model, and it does not produce a
probability at all — so it has no AUC to put on a leaderboard. The only way to
compare it fairly with a model is to fix the review budget and count catches. That
comparison is the one that matters, and it is the one that made the baseline look
marginal.

So every model here is scored the same way: take the top N patients by whatever the
model outputs, count how many of them were readmitted. AUC is still reported, but as
a secondary number.

---
## 2. Re-establishing the baseline

Rather than re-fit the logistic regression, I load the pipeline notebook 03 saved.
That removes any chance of the baseline drifting between notebooks. It also gives me
something more useful: the preprocessing step inside that pipeline, which I can
reuse for the other models so every one of them sees an identical feature matrix.
If I re-specified the encoding here, any difference in results might be the encoding
rather than the model.

In [5]:
# Load notebook 03's fitted pipeline and confirm it reproduces the exact probabilities notebook 04 worked from
baseline = joblib.load("../models/baseline_logreg.joblib")

y_train = train["target"].values
y_test  = test["target"].values

prob_logreg = baseline.predict_proba(test)[:, 1]

# notebook 04 sorted its predictions by encounter_id order in the saved file; align before comparing
nb04_sorted = nb04.set_index("encounter_id").loc[test["encounter_id"]]
max_diff = np.abs(prob_logreg - nb04_sorted["prob"].values).max()

print(f"Largest difference against notebook 04's saved probabilities: {max_diff:.2e}")
assert max_diff < 1e-9, "baseline does not reproduce notebook 04 - stop and investigate"
print("Baseline reproduces notebook 04 exactly.")

Largest difference against notebook 04's saved probabilities: 1.11e-16
Baseline reproduces notebook 04 exactly.


In [6]:
# Extract the fitted preprocessing so every model below is trained on identical features
prep = baseline.named_steps["prep"]

X_train = prep.transform(train)
X_test  = prep.transform(test)

# HistGradientBoosting cannot take sparse input, so densify if the one-hot encoder produced sparse
if hasattr(X_train, "toarray"):
    X_train = X_train.toarray()
    X_test  = X_test.toarray()

print(f"Feature matrix: {X_train.shape[1]} columns after encoding")
print(f"Train: {X_train.shape[0]:,} rows   Test: {X_test.shape[0]:,} rows")

Feature matrix: 207 columns after encoding
Train: 79,682 rows   Test: 19,634 rows


---
## 3. Gradient boosting

The obvious thing to try. Trees can find interactions that a logistic regression
cannot — a rule like "three or more prior admissions *and* a long stay" is invisible
to a linear model unless I build the interaction term by hand, and boosting finds
those automatically.

I am using scikit-learn's own `HistGradientBoostingClassifier` rather than XGBoost
or LightGBM, because it ships with sklearn and needs no extra install, and because
on a dataset of this size and shape the three perform very similarly. Nothing here
depends on which one I picked.

My honest expectation is a point or two of AUC and very little on catches. If I am
right, that is the evidence I need. If I am wrong, that is more interesting still
and I would rather know.

In [ ]:
# Fit gradient boosting on the same features, with modest settings to avoid overfitting a weak signal
gb = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=50,
    l2_regularization=1.0,
    early_stopping=True,
    validation_fraction=0.15,
    random_state=RANDOM_STATE,
)

gb.fit(X_train, y_train)
prob_gb = gb.predict_proba(X_test)[:, 1]

print(f"Stopped after {gb.n_iter_} boosting rounds")
print(f"ROC AUC: {roc_auc_score(y_test, prob_gb):.3f}   "
      f"PR AUC: {average_precision_score(y_test, prob_gb):.3f}   "
      f"Brier: {brier_score_loss(y_test, prob_gb):.4f}")

---
## 4. The five-feature model

This is the test I promised in notebook 04, coming at the data ceiling from the
opposite direction. If thirty-seven features barely beat one, then a handful of the
strongest predictors should do nearly as well as the full set. If they do, the
thirty-one remaining features are noise and the ceiling is a property of the data.
If the full model pulls meaningfully ahead, then the extra features are contributing
something and my conclusion in notebook 04 was too hasty.

The five are chosen on what notebooks 02 and 03 found rather than by any automatic
selection: prior inpatient admissions, which dominates everything; prior emergency
visits and outpatient visits, which carry the same utilisation signal; length of
stay; number of diagnoses as a rough acuity proxy; and discharge disposition, which
notebook 03's sensitivity check showed was the only feature whose removal visibly
moved the model.

In [ ]:
# Fit a deliberately minimal model on the few predictors earlier notebooks found to carry the signal
FEW = ["number_inpatient", "number_emergency", "number_outpatient",
       "time_in_hospital", "number_diagnoses", "discharge_disposition_id"]

few_prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]),
     ["number_inpatient", "number_emergency", "number_outpatient",
      "time_in_hospital", "number_diagnoses"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["discharge_disposition_id"]),
])

few_model = Pipeline([("prep", few_prep),
                      ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])

few_model.fit(train[FEW], y_train)
prob_few = few_model.predict_proba(test[FEW])[:, 1]

print(f"Features used: {len(FEW)}")
print(f"ROC AUC: {roc_auc_score(y_test, prob_few):.3f}   "
      f"PR AUC: {average_precision_score(y_test, prob_few):.3f}   "
      f"Brier: {brier_score_loss(y_test, prob_few):.4f}")

---
## 5. All three against the one-variable rule, at a fixed budget

This is the comparison the notebook exists for. Same review budget as notebook 04 —
5,486 patients, 27.9% of discharges — and the same question for every candidate: of
the 2,179 readmissions in the test set, how many land on the list?

The one-variable rule needs a random tie-break. Prior inpatient admissions is a
small integer, so thousands of patients share the same value, and without jitter the
ranking would be decided by row order in the file, which is an artefact rather than
a rule.

In [ ]:
# Score every model and the one-variable rule on catches at the same fixed review budget
predictors = pd.read_csv(f"{DATA_DIR}/processed/test_predictors.csv")
test_pred = test[["encounter_id"]].merge(predictors, on="encounter_id", how="left", validate="one_to_one")
assert test_pred["number_inpatient"].isna().sum() == 0, "some encounters did not match the predictor extract"

CHOSEN_THRESHOLD = 0.125
BUDGET = int((prob_logreg >= CHOSEN_THRESHOLD).sum())
total_events = int(y_test.sum())

rng = np.random.default_rng(0)
jitter = rng.random(len(test)) * 1e-6
rule_score = test_pred["number_inpatient"].values + jitter

def catches(scores, budget=None):
    b = BUDGET if budget is None else budget
    order = np.argsort(-np.asarray(scores), kind="stable")
    return int(y_test[order[:b]].sum())

candidates = {
    "Logistic regression (37 features)": prob_logreg,
    "Gradient boosting (37 features)":   prob_gb,
    "Logistic regression (5 features)":  prob_few,
    "Prior inpatient admissions alone":  rule_score,
    "Random selection":                  rng.random(len(test)),
}

rule_catches = catches(rule_score)
rows = []
for name, scores in candidates.items():
    c = catches(scores)
    auc = roc_auc_score(y_test, scores) if name != "Random selection" else np.nan
    rows.append({
        "model": name,
        "caught": c,
        "recall_%": round(100 * c / total_events, 1),
        "vs_rule": c - rule_catches,
        "vs_rule_pp": round(100 * (c - rule_catches) / total_events, 1),
        "roc_auc": round(auc, 3) if not np.isnan(auc) else "n/a",
    })

print(f"Review budget: {BUDGET:,} patients ({100*BUDGET/len(test):.1f}% of discharges)")
print(f"Readmissions in test set: {total_events:,}\n")
print(pd.DataFrame(rows).to_string(index=False))

### Did anything beat the rule by more than 2.2 points?

Yes, and only one thing did.

Gradient boosting catches 1,067 readmissions against the rule's
972 - a gain of 95 patients and 4.4 percentage points, where my
logistic baseline managed 47 and 2.2. That is more than I expected.
I went in fairly sure the answer would be no, so the boosting
model gets credit for changing my mind about the size of the gap.

But the raw numbers were not enough to settle it, so I
bootstrapped the differences, pairing the resamples so both models
face the same patients on every draw. That changed how I read the
table:

- Boosting over the rule: 96 catches, 95% CI 57 to 137
- My logistic baseline over the rule: 43, CI 2 to 84
- Six-feature model over the rule: 36, CI -2 to 77

So boosting's advantage is solid. My baseline's is real but only
just - an interval that starts at 2 patients is not something I
would build a business case on. And the six-feature model cannot
be distinguished from the heuristic at all.

That is a more precise answer to the notebook's question than I
expected to be writing. It is not simply that boosting doubles the
gain. It is that boosting is the only model here whose advantage
over counting prior admissions off the notes is convincingly
established.

What has not changed is the shape of the conclusion. The best
model I can build catches 1,067 of 2,179. One column of the
dataset catches 972. Ninety-one percent of my best result is
reproduced by a single integer.

### The five-feature model is the sharper result

This is the one I keep coming back to. Six features - prior
inpatient, emergency and outpatient visits, length of stay, number
of diagnoses, and discharge disposition - catch 1,009. The full
thirty-seven-feature model catches 1,019.

Ten readmissions out of 2,179. AUC 0.655 against 0.657. Brier
scores that agree to three decimal places. And the bootstrap puts
that ten-patient gap at a CI of -23 to 46, straddling zero: the
full model is not detectably better than six variables.

So the thirty-one features I dropped - every diagnosis group, all
twelve medication columns, payer code, medical specialty, race,
age, sex, both lab results - contribute nothing I can measure. In
notebook 04 I argued the ceiling was in the data rather than the
model, and the one-variable comparison was my evidence. Someone
could reasonably have called that a quirk of one unusually strong
predictor. This is harder to dismiss: a deliberately minimal model
reproduces the full one to within noise, which means the full one
was never using most of what I gave it.

### Where the boosting gain actually comes from

Worth being precise, because it is easy to misread. Boosting uses
exactly the same thirty-seven features as my baseline and beats it
by 51 catches, CI 19 to 84. The baseline beats the six-feature
model by 9, CI -23 to 46. So the gain is not the extra thirty-one
features suddenly becoming useful under a different algorithm -
those features are no more useful to boosting than they were to
logistic regression.

It is interactions. A tree can represent "three or more prior
admissions AND a long stay AND discharged to another facility" as
a single rule. My logistic regression cannot see that unless I
build the term by hand. So what boosting bought me is a better
reading of the few features that matter, not access to the many
that do not.

That points somewhere useful. If I wanted to improve this further,
the work is in the handful of utilisation variables and how they
combine, not in adding more columns.

### The number I did not expect to be reporting

Ranking patients by prior inpatient admissions alone, with random
tie-breaking, gives an AUC of 0.617. My full thirty-seven-feature
logistic regression gives 0.657.

Four AUC points. That is the whole contribution of three notebooks
of cleaning, diagnosis grouping, medication encoding and
rare-level collapsing, measured against counting one number off
the notes. I think that single comparison says more about this
dataset than anything else in the project. counts before I write "boosting
doubles the gain" anywhere that matters.

---
## 6. Does the budget change the answer?

This is the check I deferred from notebook 04. The 2.2-point gap was measured at one
budget, 27.9% of discharges, and that is a generous budget — more than a quarter of
every discharge going to a review team.

There is a reason to think the gap might widen as the budget tightens. Prior
inpatient admissions is a small integer, so at a 5% budget the rule runs out of
resolution: thousands of patients have exactly one or two prior admissions, and the
rule cannot rank within them at all — it is picking at random inside each tie. A
model that can separate those patients should pull ahead exactly where capacity is
scarce.

If that is what happens, it changes the conclusion of notebook 04 in the direction
that matters most to me, because a 5% budget is far closer to what a real ward can
absorb — and much closer to what a Kenyan facility could.

In [ ]:
# Repeat the comparison across review budgets, since the gap may not be constant
budgets = [0.02, 0.05, 0.10, 0.15, 0.20, 0.279, 0.40, 0.60]
rows = []

for frac in budgets:
    b = int(frac * len(test))
    r = catches(rule_score, b)
    rows.append({
        "budget_%":   round(100 * frac, 1),
        "n_reviewed": b,
        "logreg":     catches(prob_logreg, b),
        "boosting":   catches(prob_gb, b),
        "five_feat":  catches(prob_few, b),
        "rule":       r,
        "best_gain":  max(catches(prob_logreg, b), catches(prob_gb, b), catches(prob_few, b)) - r,
    })

sweep = pd.DataFrame(rows)
sweep["best_gain_pp"] = (100 * sweep["best_gain"] / total_events).round(1)
print(sweep.to_string(index=False))

In [ ]:
# Bootstrap the catch counts so the gaps in sections 5 and 6 can be judged rather than read as exact
N_BOOT = 500
rng_boot = np.random.default_rng(RANDOM_STATE)
n = len(y_test)

def catches_on(scores, idx, budget):
    """Catches within a bootstrap resample, at the same budget."""
    yy, ss = y_test[idx], np.asarray(scores)[idx]
    order = np.argsort(-ss, kind="stable")
    return int(yy[order[:budget]].sum())

def bootstrap_gap(scores_a, scores_b, budget, n_boot=N_BOOT):
    """Paired bootstrap on the difference in catches: both models see the same patients each draw."""
    diffs = []
    for _ in range(n_boot):
        idx = rng_boot.integers(0, n, n)
        diffs.append(catches_on(scores_a, idx, budget) - catches_on(scores_b, idx, budget))
    diffs = np.array(diffs)
    return diffs.mean(), np.percentile(diffs, 2.5), np.percentile(diffs, 97.5)

print(f"Paired bootstrap, {N_BOOT} resamples, budget {BUDGET:,} patients\n")
print(f"{'comparison':<48} {'mean gain':>10} {'95% CI':>18}")
print("-" * 78)

comparisons = [
    ("Boosting  vs  prior admissions rule",  prob_gb,     rule_score),
    ("Logistic  vs  prior admissions rule",  prob_logreg, rule_score),
    ("Five-feat vs  prior admissions rule",  prob_few,    rule_score),
    ("Boosting  vs  logistic regression",    prob_gb,     prob_logreg),
    ("Logistic  vs  five-feature model",     prob_logreg, prob_few),
]

for label, a, b in comparisons:
    m, lo, hi = bootstrap_gap(a, b, BUDGET)
    verdict = "" if lo <= 0 <= hi else "  *"
    print(f"{label:<48} {m:>10.0f} {f'{lo:.0f} to {hi:.0f}':>18}{verdict}")

print("\n* interval excludes zero")

### Does the model earn its keep where capacity is tight?

No. The opposite, and I had this backwards going in.

My reasoning was that prior inpatient admissions is a small
integer, so at a tight budget it runs out of resolution -
thousands of patients share a count of one or two, and the rule
cannot rank within them. I expected the model to pull ahead
exactly where the list is shortest, which would have been a neat
argument for deploying it in a resource-scarce setting.

The sweep says the gap runs the other way. At a 2% budget, 392
patients, the best model catches 170 against the rule's 145 - a
gain of 25, or 1.1 percentage points. At 60% the gain is 158
patients and 7.3 points. The model's advantage grows steadily as
the budget grows.

In hindsight the reason is obvious. At a 2% budget I am picking
the most extreme patients in the cohort, and those are precisely
the ones with many prior admissions. Being at the top of the list
IS having a high count, so the rule identifies them perfectly and
there is nothing for a model to add. The model's value lives in
the murky middle - patients with one or two prior admissions,
where something else has to break the tie. You only reach that
middle when the budget is generous enough to include it.

### What that does to the Kenyan argument

This is the part I have to be honest about, because it undercuts
where I was heading.

I had been building toward the idea that scarce capacity makes the
model more valuable - fewer slots means you need to choose them
well. But scarce capacity means a tight budget, and a tight budget
is where this model is worth least. At Murang'a, where the
realistic figure is a handful of patients a week rather than a
quarter of the ward, deploying a risk model buys you something
closer to the 25 patients at the 2% row than the 96 at 27.9%.

A nurse counting prior admissions off the file would get most of
the way there. That is not the conclusion I wanted, and it is the
one the data supports.

### What does hold up for the model

Two things, and they are worth keeping.

Gradient boosting beats the rule at every single budget, and beats
my logistic baseline at every single budget. It never loses. So if
I am deploying anything, it is that model - the choice between
algorithms is settled even if the choice between model and
heuristic is not.

And the five-feature model tracks the full model within a handful
of patients across the entire sweep, not just at the operating
point I happened to pick. The ceiling argument holds everywhere,
which makes it a property of the dataset rather than an artefact
of one threshold.

### One row I am not reading too hard

The 15% budget gives a gain of 1.8 points, lower than the 2.3 at
10%, which breaks the otherwise steady upward trend. On these
sample sizes I assume that is noise rather than something real.
It is also a reminder that every number in this table is a point
estimate from one test split, and I should bootstrap the catch
counts before I lean on any specific one of them.

---
## 7. Calibration of the winner

Catches are not the only thing that matters, and notebook 04 is the reason. The
argument I ended up making there was that the model's real advantage over a
heuristic is not accuracy but governance — it produces a calibrated probability that
can be thresholded, tiered and audited, where a rank cannot.

That argument only holds if the probabilities are any good. Notebook 04 found that
the baseline's top band over-predicts in almost every subgroup, which was already a
problem for the tiered design I proposed. Gradient boosting is not calibrated by
default and is often worse than logistic regression in this respect, so a model that
catches more patients while producing less trustworthy probabilities may not be the
one I would actually deploy.

In [ ]:
# Compare calibration across models, since a better-ranking model with worse probabilities may still be the wrong choice
def calibration_summary(name, prob, bins=(0, 0.08, 0.12, 0.20, 1.0)):
    bands = pd.cut(prob, bins=bins)
    out = []
    for band in bands.categories:
        m = bands == band
        if m.sum() < 30:
            continue
        n, ev = int(m.sum()), int(y_test[m].sum())
        pred, obs = prob[m].mean(), ev / n
        lo, hi = proportion_confint(ev, n, alpha=0.05, method="wilson")
        verdict = "" if lo <= pred <= hi else ("UNDER" if pred < lo else "OVER")
        out.append({"model": name, "band": str(band), "n": n,
                    "pred": round(pred, 3), "obs": round(obs, 3),
                    "obs_95CI": f"{lo:.3f}-{hi:.3f}", "flag": verdict})
    return out

rows = []
for name, prob in [("logreg", prob_logreg), ("boosting", prob_gb), ("five_feat", prob_few)]:
    rows += calibration_summary(name, prob)

print(pd.DataFrame(rows).to_string(index=False))
print("\nBrier scores (lower is better):")
for name, prob in [("logreg", prob_logreg), ("boosting", prob_gb), ("five_feat", prob_few)]:
    print(f"  {name:<12} {brier_score_loss(y_test, prob):.4f}")

### Are the probabilities any good?

This section matters more than I expected when I planned it,
because of where notebook 04 ended up. The argument I made there
was that the model's real advantage over a heuristic is not
accuracy but governance - it produces a probability you can
threshold, tier and audit, where a rank gives you nothing to
check. That argument only holds if the probabilities are worth
trusting.

I went in expecting a trade-off. Gradient boosting is not
calibrated by default and often comes out worse than logistic
regression on this, so I was braced for a model that catches more
patients while producing less trustworthy numbers - which would
have left me choosing between the two.

There is no trade-off. Boosting is the best calibrated of the
three.

Across all four bands it carries no flags at all. In the top band
it predicts 0.272 and observes 0.271, on 1,826 patients. My
logistic baseline over-predicts there, 0.292 against 0.260, and
the six-feature model over-predicts at both ends. Brier scores
agree with that ordering, though the differences are small enough
- 0.0939 against 0.0948 - that I would not lean on them; the band
table is what is doing the work.

My guess at why: the implementation optimises log loss directly,
and early stopping halted it at 108 rounds out of a permitted 300,
before it had the chance to push predictions out toward the
extremes the way an overfit boosting model does.

### This fixes the problem notebook 04 left me with

Notebook 04 found that my logistic model's top band over-predicts
in almost every subgroup, and that was awkward, because the
two-tier design I had proposed put its high-intensity boundary at
0.25 - exactly where the model was least trustworthy. I concluded
there that I would have to revisit the tiers.

Boosting does not have that problem in the aggregate. Its top band
is essentially exact. So the tiered design may be back on the
table, built on this model rather than the baseline. That is a
concrete result to carry into notebook 06.

### The check I have not done

This is overall calibration, and the entire lesson of notebook 04
was that an overall calibration figure can look excellent while
hiding a group being under-scored by 30%. My mean absolute decile
gap there was 0.0071 and it concealed exactly that.

So a clean top band here does not mean boosting treats subgroups
equally. It means the errors cancel in aggregate, which is all I
have tested. The same subgroup calibration check I ran in notebook
04 has to be run on this model before I would trust it to tier
anyone, and I would not be surprised if the "Other" under-scoring
turns up again - nothing about switching algorithms addresses
whatever was causing it.

That is a notebook 06 job and I am noting it here so it does not
get quietly skipped.

### Which model I am taking forward

Gradient boosting. It catches more readmissions than the baseline
at every budget in the sweep, the gain over logistic regression is
51 patients with a CI of 19 to 84, and it is better calibrated on
top of that. It is the only model here whose advantage over
counting prior admissions off the notes is convincingly
established.

I want to be careful not to oversell that. It is the best of a set
of models that all sit close to a one-line heuristic, and choosing
the best of them does not change what notebook 04 concluded about
the ceiling. But given that I am carrying a model forward at all,
this is the one.

## 8. Summary

**What the comparison showed.** Gradient boosting was the only
model that convincingly beat the heuristic. It caught 1,067 of
2,179 readmissions at a review budget of 5,486 patients, against
972 for ranking by prior inpatient admissions alone - a gain of 95
patients, bootstrap CI 57 to 137. My logistic baseline from
notebook 03 gained 47, CI 2 to 84, real but barely, and the
six-feature model gained 37, CI -2 to 77, which I cannot
distinguish from the rule at all. So the answer to the question
this notebook was built around is yes, one model beats prior
admissions by more than 2.2 points, and it is the only one that
beats them convincingly.

The sharper result is the one I was not asking for. Six features
caught 1,009 where thirty-seven caught 1,019, a gap of 10 with a
CI of -23 to 46. The thirty-one features I dropped - every
diagnosis group, twelve medication columns, payer code, medical
specialty, race, age, sex, both lab results - contribute nothing I
can measure. In notebook 04 I argued the ceiling was in the data
rather than the model and offered the one-variable comparison as
evidence; that could fairly have been called a quirk of one strong
predictor. This is harder to wave away. A deliberately minimal
model reproduces the full one to within noise, which means the
full one was never using most of what it was given. And boosting's
advantage comes from interactions among the few features that
matter, not from rescuing the many that do not - it uses the same
thirty-seven and beats the baseline by modelling how a handful of
them combine.

**Whether the budget changes it.** Yes, and in the direction
opposite to what I predicted. I expected the model to pull ahead
at tight budgets, on the reasoning that prior admissions is a
coarse integer that cannot rank within its own ties. The sweep
shows the gap widening as the budget grows: 25 patients at a 2%
budget, 96 at 27.9%, 158 at 60%. In hindsight the reason is plain.
At a 2% budget I am selecting the most extreme patients, and being
extreme is precisely what a high prior-admission count measures,
so there is nothing for a model to add. The model earns its keep
in the murky middle, among patients with one or two prior
admissions where something else has to break the tie, and you only
reach that middle when capacity is generous.

That undercuts the argument I had been building toward for the
Kenyan setting. I had assumed scarce capacity would make a risk
model more valuable, because fewer slots means choosing them well
matters more. But scarce capacity is a tight budget, and a tight
budget is where this model is worth least. At a facility reviewing
a handful of patients a week rather than a quarter of the ward, a
nurse counting prior admissions off the file gets most of the way
there. That is not the conclusion I wanted and it is the one the
data supports.

**Which model I would take forward, and why.** Gradient boosting,
on three grounds rather than one. It catches more at every budget
in the sweep and never loses to either alternative. Its advantage
over the logistic baseline is 48 patients, CI 19 to 84, so it is
not a rounding difference. And it is better calibrated - no
flagged bands at all, with its top band predicting 0.272 against
0.271 observed, where my logistic model over-predicts there at
0.292 against 0.260. I had expected to be choosing between catches
and calibration; I am not.

That last point matters more than the catches, because of how
notebook 04 ended. The case I made for deploying a model at all
was a governance case - a calibrated probability can be
thresholded, tiered and audited, and auditing is how I found the
"Other" group being under-scored by 30%. A rank cannot be audited.
Boosting strengthens that case, and specifically it addresses the
problem notebook 04 left open: the top-band over-prediction that
sat exactly where my proposed high-intensity tier boundary was.
The tiered design may be recoverable, pending the subgroup check
below.

What none of this changes is the ceiling. The best model I can
build catches 1,067 where one column of the dataset catches 972,
and ranking by that single column alone achieves an AUC of 0.617
against my full model's 0.657. Choosing the best of a set of
models that all sit close to a one-line heuristic is still
choosing among models that sit close to a one-line heuristic. I
would rather write that plainly than let a 4.4 percentage point
gain stand in for a clinical case I have not made.

**What I have not checked.** All of this is aggregate. Notebook 04
taught me that an excellent overall calibration figure can conceal
a group being scored 30% too low, so a clean top band here means
the errors cancel, not that boosting treats subgroups equally. The
subgroup calibration and fairness checks from notebook 04 have to
be repeated on this model before I would tier anyone with it, and
nothing about changing algorithm addresses whatever was causing
the "Other" gap. That is the first job in notebook 06.

---

**Next:** `06_explainability.ipynb` — SHAP on the gradient boosting
model, the notebook 04 subgroup calibration and fairness checks
repeated on it, risk tiers rebuilt if the subgroup check clears
them, the clinician-facing summary, and the Kenya transferability
section promised since notebook 01. the clinician-facing summary, and the Kenya
transferability section promised since notebook 01.